# Booleans of the native BREP

The booleans of `gbs.brep` (`fuse`, `cut`, `common`) on primitive solids built with the BREP API itself, shown in 3D: the operands in transparency (blue: a, orange: b), then each result with **one color per face**, so that the splits of the faces show, and the **edges of the BREP** in black. The seams where a periodic surface (cylinder, cone, sphere, torus) closes on itself are not drawn: they bound no region (`ex.add_shape(..., seams=True)` shows them).

The cases are those of the comparison with OpenCascade (`gbs-occt/tests/tests_brep2_occt.cpp`): generic intersections, coincident faces, tangent contacts, branch points, poles. The same cases are rendered off screen by `gallery.py` (the CI publishes the gallery as an artifact).

In [ ]:
import os
import pyvista as pv
import gbs_examples as ex

brep = ex.brep
# interactive (trame), or images only (static); GBS_NOTEBOOK_BACKEND overrides it (the tests run "static")
try:
    pv.set_jupyter_backend(os.environ.get("GBS_NOTEBOOK_BACKEND", "trame"))
except Exception:
    pv.set_jupyter_backend("static")

## The cases

In [ ]:
for case in ex.CASES:
    print(f"{case.name:24s} {case.title}" + (f"  ({case.note})" if case.note else ""))

## One case: operands and the three results

Change `name` to any case above. Rotating one view rotates them all.

In [ ]:
name = "box_sphere"
case = next(c for c in ex.CASES if c.name == name)
result = ex.run(case)
for op, shape in result.shapes.items():
    report = brep.check(result.model, shape)
    print(op, "valid" if report.ok() else report, len(brep.explore_faces(result.model, shape)), "faces")
for op, message in result.errors.items():
    print(op, "error:", message)
plotter, meshes = ex.plot_case(result, case)
plotter.show()

## A result alone, larger

Each face of the result in its own color, the edges of the BREP in black.

In [ ]:
p = pv.Plotter()
ex.add_shape(p, result.model, result.shapes["cut"])
p.show()

## Building your own case

The primitives: `ex.box(m, origin, size)`, `ex.cylinder(m, radius, height, base, axis)`, `ex.cone(m, radius, height, base, axis)`, `ex.sphere(m, radius, center, axis)`, `ex.torus(m, major, minor, center, axis)`; `exact=False` builds the NURBS of the circles instead of exact primitives.

In [ ]:
m = brep.Model()
a = ex.cylinder(m, 1.0, 4.0, (0, 0, -2))
b = ex.torus(m, 1.5, 0.4, (0, 0, 0), (1, 0, 0))
mine = ex.Case("mine", "Cylinder and torus", lambda model: (a, b))
r = ex.Result(m, a, b)
for op, f in ex.OPERATIONS.items():
    try:
        r.shapes[op] = f(m, a, b).shape
    except brep.BRepError as e:
        r.errors[op] = str(e)
plotter, meshes = ex.plot_case(r, mine)
plotter.show()